In [1]:
import torch
from torch.utils.data import DataLoader, Subset
from transformers import T5TokenizerFast, CLIPProcessor, CLIPTokenizerFast, CLIPImageProcessorFast, T5ForConditionalGeneration


from peft import LoraConfig, get_peft_model, TaskType
from torch.amp import autocast, GradScaler
from tqdm import tqdm
import os 


In [2]:
from Modules.config import (TRAIN_IMAGE_DIR,
                            TEST_IMAGE_DIR,
                            FAISS_IMAGE_PATH,
                            TRAIN_METADATA_PATH,
                            TEST_METADATA_PATH,
                            CLIP_MODEL_NAME,
                            T5_MODEL_NAME,
                            VLM_CHECKPOINT_DIR,
                            T5_DECODER_LORA_CONFIG)

from Modules.FusionVLM import FusionVLM, create_default_FusionVLM, load_default_FusionVLM, save_FusionVLM
from Modules.retrieval_module import Retriever
from Modules.datasets import VLMDataset, VLMDataCollator
from Modules.utils import print_model_param_stats, add_dict
from Modules.metrics import evaluate_captioning, setup_nltk
from Modules.train_VLM import train_and_evaluate_model

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [4]:
CLIP_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME, use_fast=True, local_files_only=True)
# CLIP_tokenizer = CLIPTokenizerFast.from_pretrained("openai/clip-vit-base-patch32")
T5_tokenizer = T5TokenizerFast.from_pretrained(T5_MODEL_NAME, local_files_only=True)
collator = VLMDataCollator(CLIP_processor, T5_tokenizer, device=DEVICE)

In [5]:
retriever = Retriever(metadata_path=TRAIN_METADATA_PATH, faiss_path=FAISS_IMAGE_PATH)

train_dataset = VLMDataset(image_dir=TRAIN_IMAGE_DIR, 
                           ref_image_dir=TRAIN_IMAGE_DIR,
                           metadata_path=TRAIN_METADATA_PATH,
                           retriever=retriever)

test_dataset = VLMDataset(image_dir=TEST_IMAGE_DIR, 
                           ref_image_dir=TRAIN_IMAGE_DIR,
                           metadata_path=TEST_METADATA_PATH,
                           retriever=retriever)

In [6]:
BATCH_SIZE = 16
NUM_WORKERS = 0

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

In [7]:
# import numpy as np
# from torch.utils.data import DataLoader, Subset

# num_train_samples = 1024
# num_test_samples = 128


# indices = np.random.choice(len(train_dataset), num_train_samples, replace=False)
# train_subset = Subset(train_dataset, indices)
# train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

# indices = np.random.choice(len(test_dataset), num_test_samples, replace=False)
# test_subset = Subset(test_dataset, indices)
# test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

In [8]:
model = create_default_FusionVLM().to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
# print(f"Total parameters: {num_params:,}\nText Decoder", end=' ')
# model.text_decoder.print_trainable_parameters()

AttributeError: 'Embedding' object has no attribute 'modules_to_save'

In [ ]:
from Modules.FusionVLM import apply_lora_config

In [ ]:
model = FusionVLM(vision_encoder_name=CLIP_MODEL_NAME,
                    text_encoder_name=T5_MODEL_NAME,
                    T5_text_decoder_name=T5_MODEL_NAME,
                    num_fusion_blocks=4,
                    use_local_files=True
                    )


In [ ]:
model = apply_lora_config(model).to(DEVICE)

In [ ]:
print_model_param_stats(model)

In [ ]:
print_model_param_stats(model)

In [ ]:
NUM_EPOCHS = 2
full_history = {}
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=0.01
)

setup_nltk()
os.makedirs(VLM_CHECKPOINT_DIR, exist_ok=True)

In [ ]:
history = train_and_evaluate_model(model, train_loader, optimizer, NUM_EPOCHS, test_loader, T5_tokenizer)
add_dict(full_history, history)

In [ ]:
save_FusionVLM(model, f'epoch2', VLM_CHECKPOINT_DIR)

In [ ]:
full_history

In [ ]:
import json
with open('history.json', "w", encoding="utf-8") as f:
    json.dump(full_history, f, indent=2, ensure_ascii=False)

In [ ]:
with torch.no_grad():
    for batch in test_loader:
        gt_captions = batch["all_captions"]  # List[List[str]]

        generated_ids = model.generate(
            query_pixel_values=batch["query_pixel_values"],
            retrieved_pixel_values=batch["retrieved_pixel_values"],
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            max_length=64,
            num_beams=1,
            # do_sample=True,
            # top_p=0.9,
            # temperature=0.8,
            # repetition_penalty=1.2,
        )

        decoded = T5_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        for i in range(len(decoded)):
            print(gt_captions[i])
            print(decoded[i])            
        break